In [1]:
email_conversation = """From: 테디 (teddy@teddynote.com)
To: 이은채 대리님 (eunchae@teddyinternational.me)
Subject: RAG 솔루션 시연 관련 미팅 제안

안녕하세요, 이은채 대리님,

저는 테디노트의 테디입니다. 최근 귀사에서 AI를 활용한 혁신적인 솔루션을 모색 중이라는 소식을 들었습니다. 테디노트는 AI 및 RAG 솔루션 분야에서 다양한 경험과 노하우를 가진 기업으로, 귀사의 요구에 맞는 최적의 솔루션을 제공할 수 있다고 자부합니다.

저희 테디노트의 RAG 솔루션은 귀사의 데이터 활용을 극대화하고, 실시간으로 정확한 정보 제공을 통해 비즈니스 의사결정을 지원하는 데 탁월한 성능을 보입니다. 이 솔루션은 특히 다양한 산업에서의 성공적인 적용 사례를 통해 그 효과를 입증하였습니다.

귀사와의 협력 가능성을 논의하고, 저희 RAG 솔루션의 구체적인 기능과 적용 방안을 시연하기 위해 미팅을 제안드립니다. 다음 주 목요일(7월 18일) 오전 10시에 귀사 사무실에서 만나 뵐 수 있을까요?

미팅 시간을 조율하기 어려우시다면, 편하신 다른 일정을 알려주시면 감사하겠습니다. 이은채 대리님과의 소중한 만남을 통해 상호 발전적인 논의가 이루어지길 기대합니다.

감사합니다.

테디
테디노트 AI 솔루션팀"""

In [2]:
from pydantic import BaseModel, Field

# 이메일 본문으로부터 주요 엔티티 추출 
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    company: str = Field(description="메일 보낸 사람의 회사 정보")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")


In [3]:
## LCEL 구조 

# chain = prompt | llm | output_parser

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

In [6]:
from langchain_core.output_parsers import PydanticOutputParser

# PydanticOutputParser 생성
output_parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [9]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

#QUESTION:
다음의 이메일 내용 중에서 주요 내용을 추출해 주세요.

#EMAIL CONVERSATION:
{email_conversation}

#FORMAT:
{format}
"""
)

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=output_parser.get_format_instructions())

In [10]:
print(output_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "company": {"description": "메일 보낸 사람의 회사 정보", "title": "Company", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "company", "email", "

In [12]:
prompt

PromptTemplate(input_variables=['email_conversation'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "company": {"description": "메일 보낸 사람의 회사 정보", "title": "Company", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description

In [13]:
# 체인 생성
chain = prompt | llm | output_parser

In [14]:
# 체인 실행
answer = chain.invoke({"email_conversation": email_conversation})

In [15]:
print(answer)

person='테디' company='테디노트' email='teddy@teddynote.com' subject='RAG 솔루션 시연 관련 미팅 제안' summary='테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 테디노트의 솔루션이 귀사의 데이터 활용을 극대화하고 비즈니스 의사결정을 지원할 수 있음을 강조하고 있습니다. 미팅은 다음 주 목요일 오전 10시에 제안되었습니다.' date='7월 18일 오전 10시'


In [16]:
answer.email

'teddy@teddynote.com'

### 검색: SERP API

In [17]:
!pip install google-search-results

In [18]:
import os 

os.environ["SERPAPI_API_KEY"] = "1e5169c8c8079fbca6ccf5067b2eab8558a23d35453cec51c471cda99aaba39e"

In [19]:
from langchain_community.utilities import SerpAPIWrapper


In [20]:

params = {"engine": "google", "gl": "kr", "hl": "ko", "num": "3"}

search = SerpAPIWrapper(params=params)

In [21]:
search

SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'gl': 'kr', 'hl': 'ko', 'num': '3'}, serpapi_api_key='1e5169c8c8079fbca6ccf5067b2eab8558a23d35453cec51c471cda99aaba39e', aiosession=None)

In [22]:
search.run("테디노트")

'[\'데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \\u200d♂️ ...more ...more fastcampus.co.kr/data_online_teddyand 2 ...\', \'자동화된 메타데이터 태깅으로 문서의 메타데이터(metadata) 생성 및 자동 라벨링 ... 문서 관리를 위한 메타데이터 태깅은 필수적이지만 번거로울 수 있습니다. OpenAI 기반 ...\', \'블로그와 유튜브 "테디노트"를 운영하고 있으며, "파이썬 딥러닝 텐서플로"를 집필하였습니다. 데이터분석과 AI를 사랑하고 지식공유에 활발히 참여하고 있습니다.\']'

In [23]:
search.run("테디노트 site:github.com")

"['✔️ 코드스테이츠 X 테디노트 - 깃헙 블로그 제작하기 강의. ✔️ SK그룹 - 2023년 상반기 텐서플로우 딥러닝 과정 강의. ✔️ S-Oil - 파이썬 데이터 분석, 머신러닝 과정 강의.', '차분하게 배워볼 수 있는 유튜브 채널. 테디노트. 텐서플로우 관련 영상들이 주를 이룹니다. 데이터 분석, 머신러닝, 그리고 딥러닝 주제를 다루는 유튜브 채널.']"

In [24]:
answer.email

'teddy@teddynote.com'

In [27]:
query = f"{answer.person} {answer.company} {answer.email}"
query

'테디 테디노트 teddy@teddynote.com'

In [33]:
search_result = search.run(query)


In [32]:
type(search_result)

str

In [38]:
search_result = eval(search_result)

In [39]:
type(search_result)

list

In [ ]:
# 검색 결과
search_result_string = '\n'.join(search_result)

In [45]:
answer

EmailSummary(person='테디', company='테디노트', email='teddy@teddynote.com', subject='RAG 솔루션 시연 관련 미팅 제안', summary='테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 테디노트의 솔루션이 귀사의 데이터 활용을 극대화하고 비즈니스 의사결정을 지원할 수 있음을 강조하고 있습니다. 미팅은 다음 주 목요일 오전 10시에 제안되었습니다.', date='7월 18일 오전 10시')

In [84]:
from langchain.prompts import PromptTemplate

report_prompt = PromptTemplate.from_template("""
당신은 이메일의 주요 정보를 바탕으로 요약 정리해 주는 전문가입니다.
당신의 임무는 다음의 이메일 정보를 바탕으로 고보서 형식의 요약을 작성하는 것입니다.
주어진 정보를 기반으로 양식(format)에 맞춰서 요약을 작성해 주세요.
답변에는 카테고리별로 emoji를 적극 활용하여 답변해 주세요.

#Information:
- Name: {sender}
- Additional Information about sender: {additional_information}
- Company: {company}
- Email: {email}
- Subject: {subject}
- Summary: {summary}
- Date: {date}

#Format(in markdown format):
🚗보낸 사람:
- (보낸 사람의 이름, 회사 정보)

💚이메일 주소:
- (보낸 사람의 이메일 주소)

💥보낸 사람과 관련하여 검색된 추가 정보: 
- (검색된 정보)

주요 정보: 
- (이메일 제목, 요약)

일정:
- (미팅 날짜 및 시간)

#Answer:
""")



In [85]:
from langchain_core.output_parsers import StrOutputParser

report_chain = report_prompt | ChatOpenAI(model="gpt-4-turbo", temperature=0) | StrOutputParser()

In [86]:
report_response = report_chain.invoke(
  {
    "sender": answer.person,
    "additional_information": search_result_string,
    "company": answer.company,
    "email": answer.email,
    "subject": answer.subject,
    "summary": answer.summary,
    "date": answer.date, 
  }
)

In [88]:
print(report_response)

🚗보낸 사람:
- 테디, 테디노트

💚이메일 주소:
- teddy@teddynote.com

💥보낸 사람과 관련하여 검색된 추가 정보: 
- 테디노트 X 패스트캠퍼스 "RAG 비법노트" · 환경 설정 (Mac) · 환경 설정 (Windows). LocalModels. GGUF · HuggingFace gguf 파일을 Ollama 로딩 · TeddyNote. 데이터와 인공지능을 좋아하는 개발자 노트 · 검색. 토글 메뉴. 카테고리 · 태그 · 연도 · 강의 · 어바웃미 · Teddy. Creator & Data Lover. 팔로우. Pangyo, 데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다.

주요 정보: 
- 제목: RAG 솔루션 시연 관련 미팅 제안
- 요약: 테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 테디노트의 솔루션이 귀사의 데이터 활용을 극대화하고 비즈니스 의사결정을 지원할 수 있음을 강조하고 있습니다.

일정:
- 7월 18일 오전 10시
